# "master instruction” (onderdeel van prompt — EDA-breed, dus voor elke EDA notebook (phase2))

Je werkt als senior research assistant voor een masterthesis in data analysis. Voor meer informatie over de thesis / onderzoeksvoorstel / opzet: bekijk bron "Geannoteerd_onderzoeksvoorstel.md" en voor extra gelinkte literatuur bron; "External factors and SHAP in Urban Parking copy.pdf" (beide bestanden zijn te vinden in de map 'literatuur_en_info' (binnen dit project)) 

voor structuur en gewenste flow check projectalomvattede: "README.md"

Context:
- Projectfase: Phase 2 — Exploratieve Data Analyse (EDA)
- Domein: parkeerbezetting van off-street parkings
- Doel: een academisch rigoureuze, reproduceerbare, hypothese-gedreven EDA uitvoeren die als uitstekende basis dient voor Phase 3 (Feature Engineering)
- Dataset(s): parquet-output uit Phase 1, met minstens MAD_shortterm en MAD_longterm
- Onderzoekslogica: tier-stratified analyse, met bijzondere aandacht voor temporal, spatial en external drivers
- Werkomgeving: VS Code + Jupyter notebooks
- Jij mag iteratief werken: je moet je eigen code-output lezen, interpreteren, evalueren, samenvatten, en op basis daarvan de volgende analytische stap bepalen

Belangrijke werkinstructies:
1. Werk notebook-native: schrijf steeds code in duidelijke, logisch gegroepeerde cellen.
2. Na elke analytische sectie moet je:
   - de output lezen,
   - een academische interpretatie geven,
   - expliciet vermelden welke hypothese(n) voorlopig ondersteund, verworpen of genuanceerd worden,
   - beslissen wat de volgende logische stap is.
3. Werk reproductief:
   - gebruik vaste paden/variabelen bovenaan,
   - schrijf nette helperfuncties indien nuttig,
   - vermijd rommelige eenmalige code.
4. Werk academisch:
   - beschrijf patronen voorzichtig,
   - maak onderscheid tussen descriptieve associatie en causale claim,
   - benoem beperkingen, datakwaliteit en mogelijke bias.
5. Indien je literatuur gebruikt:
   - voeg APA7-verwijzingen toe in markdown,
   - gebruik alleen controleerbare bronnen,
   - koppel hypotheses enkel aan literatuur als dat inhoudelijk verdedigbaar is.
6. Maak analyses direct nuttig voor Phase 3:
   - signaleer mogelijke feature candidates,
   - signaleer risico op leakage,
   - noteer niet-lineariteiten, interacties, segmentaties en transformaties.
7. Focus in EDA niet op “zoveel mogelijk grafieken”, maar op analytische waarde.
8. Rapporteer steeds ook wat NIET overtuigend blijkt.
9. Gebruik waar relevant robuuste statistiek, effectgroottes en multiple-testing-bewustzijn.
10. Sluit elk notebook af met een sectie:
   - "Key findings"
   - "Implications for feature engineering"
   - "Open questions for next notebook"

Wanneer je literatuur gebruikt om een hypothese te motiveren:
- gebruik alleen bronnen die inhoudelijk echt passen bij parkeerbezetting, mobiliteit, weersinvloeden, events, forecasting of XAI;
- label speculatieve hypothesen expliciet als speculatief maar toetsbaar;
- geef APA7-verwijzingen in markdown;
- vermijd het doen alsof literatuur causale evidentie levert wanneer het eigenlijk om associatieve studies gaat;
- als de data de literatuur niet ondersteunen, rapporteer dat eerlijk.

Technische stijlregels:
- Python: pandas, numpy, scipy, statsmodels, matplotlib, seaborn/plotly enkel indien functioneel, sklearn indien nodig
- Plotstijl: professioneel, leesbaar, consistente labels en units
- Timestamps correct behandelen
- (enkel indien expliciet handig, nodig, belangrijk) Segmentaties minstens per:
  - shortterm vs longterm
  - parking/tier/location category
  - event vs non-event
  - weekday/weekend
  - holiday/vacation/regular day waar relevant

Schrijf elke interpretatieve markdown-sectie alsof ze later kan worden herwerkt tot tekst voor de masterthesis.

Stijlregels:
- helder, academisch, voorzichtig
- geen losse bullet dump als lopende tekst beter is
- benoem richting, grootteorde, onzekerheid en beperking
- maak expliciet waarom het resultaat relevant is voor de volgende fase

Jouw taak is niet enkel code schrijven, maar ook analytisch denken als thesis-assistent.

## Notebook specifieke prompt:
Maak deze notebook `eda_00_protocol_and_data_audit.ipynb`.

Doel:
Een strikte EDA-start maken met data-audit, schema-validatie, datakwaliteit, coverage, missingness, flags en analyseprotocol.

Beschikbare datasets:
- MAD_shortterm.parquet
- MAD_longterm.parquet

Voer dit uit:
1. Laad beide datasets.
2. Toon shape, kolommen, dtypes, datumrange, aantal unieke parkings, verdeling over jaren.
3. Controleer of verwachte kernkolommen aanwezig zijn.
4. Analyseer missingness systematisch:
   - overall
   - per jaar
   - per parking
   - per featuregroep (temporal, occupancy, weather, calendar, spatial, events, quality flags)
5. Analyseer datakwaliteit:
   - occupancy_rate buiten [0,1]
   - overcapacity
   - inconsistent occupancy
   - frozen sensor
   - blackout / low_data_coverage / partial_year
   - QC-kolommen en hun beschikbaarheid
6. Maak een coverage-overzicht:
   - observaties per jaar
   - observaties per parking
   - uurdekking
   - gaten in tijdsreeksen
7. Geef een academische interpretatie:
   - welke beperkingen zijn kritisch?
   - welke subsets zijn veiliger voor inferentie?
   - waar moet EDA later gevoeligheidsanalyses doen?
8. Definieer expliciet een EDA-analyseprotocol:
   - welke observaties blijven standaard in scope?
   - welke flags worden standaard gerapporteerd?
   - welke analyses gebeuren descriptief op volledige data en welke op strictere filtered subsets?
9. Sluit af met:
   - "Data audit conclusions"
   - "Risks for interpretation"
   - "Implications for downstream EDA and feature engineering"

Belangrijk:
- Maak een duidelijk onderscheid tussen shortterm en longterm.
- Gebruik markdown-cellen om je interpretatie te schrijven.
- Als je iets verdachts vindt, voer meteen een extra diagnostische analyse uit in plaats van enkel te signaleren.
- Laat je volgende stap afhangen van de output.

## 0. Setup en data loading
We laden beide MAD-datasets vanuit `data_processed/` met vaste paden zodat de audit volledig reproduceerbaar blijft.

In [1]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 200)


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "data_processed").exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError("Project root not found from current working directory")


PROJECT_ROOT = find_project_root()

# === AUTO-EXPORT ARTIFACTS (figures + displayed tables) ===
NOTEBOOK_SLUG = "eda_00_protocol_and_data_audit"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts" / "phase2" / NOTEBOOK_SLUG
FIG_DIR = ARTIFACTS_DIR / "figures"
TABLE_DIR = ARTIFACTS_DIR / "tables"
LOG_DIR = ARTIFACTS_DIR / "logs"

for _d in [ARTIFACTS_DIR, FIG_DIR, TABLE_DIR, LOG_DIR]:
    _d.mkdir(parents=True, exist_ok=True)


def _safe_artifact_name(name: str) -> str:
    allowed = set("abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789_-")
    s = "".join(ch if ch in allowed else "_" for ch in str(name))
    while "__" in s:
        s = s.replace("__", "_")
    s = s.strip("_")
    return s or "artifact"


def save_dataframe_artifact(df: pd.DataFrame, name: str, index: bool = True) -> dict[str, str | None]:
    base = _safe_artifact_name(name)
    csv_path = TABLE_DIR / f"{base}.csv"
    parquet_path = TABLE_DIR / f"{base}.parquet"

    df.to_csv(csv_path, index=index)
    parquet_ok = True
    try:
        df.to_parquet(parquet_path, index=index)
    except Exception:
        parquet_ok = False

    return {
        "csv": str(csv_path),
        "parquet": str(parquet_path) if parquet_ok else None,
    }


if not globals().get("_DISPLAY_AUTO_EXPORT_PATCHED", False):
    _DISPLAY_AUTO_EXPORT_PATCHED = True
    _ORIG_DISPLAY = display
    _DISPLAY_COUNTER = {"n": 0}

    def display(*objs, **kwargs):
        for obj in objs:
            if isinstance(obj, pd.DataFrame):
                _DISPLAY_COUNTER["n"] += 1
                save_dataframe_artifact(obj, f"display_{_DISPLAY_COUNTER['n']:03d}", index=True)
        return _ORIG_DISPLAY(*objs, **kwargs)


try:
    import matplotlib.pyplot as plt  # noqa: F401

    if not getattr(plt, "_AUTO_EXPORT_PATCHED", False):
        _ORIG_PLT_SHOW = plt.show
        _FIG_COUNTER = {"n": 0}

        def _show_and_save(*args, **kwargs):
            fig_nums = list(plt.get_fignums())
            for fig_num in fig_nums:
                fig = plt.figure(fig_num)
                _FIG_COUNTER["n"] += 1
                fig_path = FIG_DIR / f"fig_{_FIG_COUNTER['n']:03d}.png"
                fig.savefig(fig_path, dpi=150, bbox_inches="tight")
            return _ORIG_PLT_SHOW(*args, **kwargs)

        plt.show = _show_and_save
        plt._AUTO_EXPORT_PATCHED = True
    FIG_EXPORT_ENABLED = True
except Exception:
    FIG_EXPORT_ENABLED = False

print(f"Artifacts directory: {ARTIFACTS_DIR}")
print(f"- Figures: {FIG_DIR}")
print(f"- Tables: {TABLE_DIR}")

DATA_DIR = PROJECT_ROOT / "data_processed"

DATA_PATHS = {
    "shortterm": DATA_DIR / "MAD_shortterm.parquet",
    "longterm": DATA_DIR / "MAD_longterm.parquet",
}

for name, path in DATA_PATHS.items():
    if not path.exists():
        raise FileNotFoundError(f"Dataset ontbreekt: {name} -> {path}")

dfs = {name: pd.read_parquet(path).copy() for name, path in DATA_PATHS.items()}

for name, df in dfs.items():
    print(f"{name}: {df.shape[0]:,} rijen x {df.shape[1]:,} kolommen")

Artifacts directory: /Users/emilevandevoorde/Documents/mp_mechelen_parking_v2/artifacts/phase2/eda_00_protocol_and_data_audit
- Figures: /Users/emilevandevoorde/Documents/mp_mechelen_parking_v2/artifacts/phase2/eda_00_protocol_and_data_audit/figures
- Tables: /Users/emilevandevoorde/Documents/mp_mechelen_parking_v2/artifacts/phase2/eda_00_protocol_and_data_audit/tables
shortterm: 284,524 rijen x 66 kolommen
longterm: 46,643 rijen x 65 kolommen


**Interpretatie (initiele laadstap)**

Beide datasets zijn succesvol ingeladen. We kunnen nu expliciet shortterm en longterm scheiden in elke auditstap.

**Hypothese status**
- H0: De basisbestanden zijn beschikbaar en technisch leesbaar. **Ondersteund**.

**Volgende logische stap**
- Eerst een structureel profiel (shape, kolommen, dtypes, datumrange, parkings, jaarverdeling) om de auditbasis vast te leggen.

## 1. Structuurprofiel en schema-overzicht

In [2]:
def dataset_profile(df: pd.DataFrame, name: str) -> pd.Series:
    return pd.Series(
        {
            "dataset": name,
            "n_rows": int(df.shape[0]),
            "n_cols": int(df.shape[1]),
            "date_min": pd.to_datetime(df["rounded_hour"], errors="coerce").min(),
            "date_max": pd.to_datetime(df["rounded_hour"], errors="coerce").max(),
            "n_unique_parkings": int(df["parking_id"].nunique(dropna=True)),
        },
        name=name,
    )


profiles = pd.DataFrame({name: dataset_profile(df, name) for name, df in dfs.items()}).T
display(profiles)

year_distribution = pd.concat(
    {name: df["year"].value_counts().sort_index() for name, df in dfs.items()},
    axis=1,
).fillna(0).astype(int)
year_distribution.index.name = "year"
display(year_distribution)

for name, df in dfs.items():
    print(f"\n{name.upper()} - dtypes")
    dtype_table = df.dtypes.astype(str).rename("dtype").to_frame()
    display(dtype_table)

,dataset,n_rows,n_cols,date_min,date_max,n_unique_parkings
shortterm,shortterm,284524,66,2018-12-31 23:00:00,2026-02-02 00:00:00,10
longterm,longterm,46643,65,2024-01-01 00:00:00,2024-12-31 22:00:00,7


,shortterm,longterm
year,,
2018,5,0
2019,33000,0
2020,32741,0
2021,13630,0
2022,12762,0
2023,39980,0
2024,57116,46643
2025,87600,0
2026,7690,0



SHORTTERM - dtypes


,dtype
parking_id,str
parking_id_hash,category
parking_type,str
kind,str
year,int32
month,int32
date_only,datetime64[ms]
rounded_hour,datetime64[ns]
hour,int32
weekday_int,int32



LONGTERM - dtypes


,dtype
parking_id,str
parking_id_hash,category
parking_type,str
kind,str
year,int32
month,int32
date_only,datetime64[ms]
rounded_hour,datetime64[ns]
hour,int32
weekday_int,int32


**Interpretatie**

De shortterm-dataset heeft een brede meerjarige dekking (2018-2026), terwijl longterm enkel 2024 bevat. Dit impliceert dat trend- of regimevergelijkingen over meerdere jaren hoofdzakelijk op shortterm kunnen gebeuren.

**Hypothese status**
- H1: Shortterm en longterm hebben een vergelijkbare datastructuur. **Genuanceerd ondersteund** (zeer gelijkaardig schema, maar niet identiek).

**Volgende logische stap**
- Formele validatie van verwachte kernkolommen en expliciete kolomverschillen tussen beide datasets.

## 2. Kernkolommen en schema-validatie

In [3]:
CORE_COLUMNS_REQUIRED = [
    "parking_id",
    "parking_type",
    "kind",
    "year",
    "month",
    "date_only",
    "rounded_hour",
    "hour",
    "number_of_spaces_override",
    "number_of_occupied_spaces",
    "occupancy_rate",
    "flag_occ_inconsistent",
    "flag_frozen_sensor",
    "low_data_coverage",
    "system_blackout",
    "partial_year",
]

CORE_COLUMNS_OPTIONAL = [
    "flag_overcapacity",
    "total_capacity",
    "qc_temp",
    "qc_precip",
    "qc_wind_speed",
    "qc_wind_gusts",
    "qc_humidity",
    "qc_pressure",
]

presence_required = pd.DataFrame(index=CORE_COLUMNS_REQUIRED)
presence_optional = pd.DataFrame(index=CORE_COLUMNS_OPTIONAL)

for name, df in dfs.items():
    presence_required[name] = presence_required.index.isin(df.columns)
    presence_optional[name] = presence_optional.index.isin(df.columns)

presence_required["all_datasets"] = presence_required.all(axis=1)
presence_optional["all_datasets"] = presence_optional.all(axis=1)

display(presence_required)
display(presence_optional)

shortterm_only = sorted(set(dfs["shortterm"].columns) - set(dfs["longterm"].columns))
longterm_only = sorted(set(dfs["longterm"].columns) - set(dfs["shortterm"].columns))

print("Kolommen enkel in shortterm:", shortterm_only)
print("Kolommen enkel in longterm:", longterm_only)

,shortterm,longterm,all_datasets
parking_id,True,True,True
parking_type,True,True,True
kind,True,True,True
year,True,True,True
month,True,True,True
date_only,True,True,True
rounded_hour,True,True,True
hour,True,True,True
number_of_spaces_override,True,True,True
number_of_occupied_spaces,True,True,True


,shortterm,longterm,all_datasets
flag_overcapacity,True,False,False
total_capacity,True,True,True
qc_temp,True,True,True
qc_precip,True,True,True
qc_wind_speed,True,True,True
qc_wind_gusts,True,True,True
qc_humidity,True,True,True
qc_pressure,True,True,True


Kolommen enkel in shortterm: ['flag_overcapacity']
Kolommen enkel in longterm: []


**Interpretatie**

Alle vereiste kernkolommen zijn aanwezig in beide datasets. Het enige schema-verschil is `flag_overcapacity`, die enkel in shortterm aanwezig is. Overcapacity moet daarom in longterm afgeleid worden uit `number_of_occupied_spaces > number_of_spaces_override`.

**Hypothese status**
- H2: Kernschema is audit-geschikt voor beide datasets. **Ondersteund**.

**Volgende logische stap**
- Systematische missingness-analyse (overall, per jaar, per parking en per featuregroep).

## 3. Missingness-audit

In [4]:
FEATURE_GROUPS = {
    "temporal": [
        "year",
        "month",
        "date_only",
        "rounded_hour",
        "hour",
        "weekday_int",
        "weekday_name",
        "day_type",
    ],
    "occupancy": [
        "number_of_spaces_override",
        "number_of_occupied_spaces",
        "occupancy_rate",
    ],
    "weather": [
        "temp_c",
        "precip_mm",
        "wind_speed_ms",
        "wind_gusts_ms",
        "humidity_pct",
        "pressure_hpa",
        "sun_duration_min",
        "shortwave_wm2",
        "sun_intensity_wm2",
        "qc_temp",
        "qc_precip",
        "qc_wind_speed",
        "qc_wind_gusts",
        "qc_humidity",
        "qc_pressure",
        "humidity_suspect",
        "sun_duration_inconsistent",
        "precip_imputed",
    ],
    "calendar": [
        "national_holiday_name",
        "other_holiday_name",
        "vacation_name",
        "vacation_type",
        "is_national_holiday",
        "is_other_holiday",
        "is_any_holiday",
        "is_school_vacation",
        "calendar_day_class",
    ],
    "spatial": [
        "parking_id",
        "parking_id_hash",
        "parking_type",
        "kind",
        "longitude",
        "latitude",
        "total_capacity",
        "parking_location_category",
    ],
    "events": [
        "event_scale_max",
        "n_concurrent_events",
        "football_kickoff_hour",
        "data_confidence",
        "event_names_combined",
        "is_football_day",
        "is_sport_day",
        "is_festival_day",
        "is_procession_day",
        "is_kermis_day",
        "is_markt_day",
        "is_carnival_day",
        "is_other_day",
        "is_event_day",
    ],
    "quality_flags": [
        "flag_occ_inconsistent",
        "flag_overcapacity",
        "flag_frozen_sensor",
        "low_data_coverage",
        "system_blackout",
        "partial_year",
    ],
}


def feature_group_missingness(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for group, cols in FEATURE_GROUPS.items():
        present = [c for c in cols if c in df.columns]
        missing = [c for c in cols if c not in df.columns]
        miss_pct = float(df[present].isna().mean().mean() * 100) if present else np.nan
        rows.append(
            {
                "feature_group": group,
                "missing_pct": miss_pct,
                "n_present_columns": len(present),
                "missing_columns": ", ".join(missing) if missing else "",
            }
        )
    return pd.DataFrame(rows).sort_values("missing_pct", ascending=False)


for name, df in dfs.items():
    print(f"\n=== {name.upper()} | Overall missingness (top 20) ===")
    overall = (df.isna().mean() * 100).sort_values(ascending=False).head(20).round(2).rename("missing_pct")
    display(overall.to_frame())

    print(f"\n=== {name.upper()} | Missingness per jaar (gemiddeld % missende velden per rij) ===")
    per_year = (
        df.assign(_row_missing=df.isna().mean(axis=1))
        .groupby("year")["_row_missing"]
        .mean()
        .mul(100)
        .round(2)
        .rename("avg_row_missing_pct")
    )
    display(per_year.to_frame())

    print(f"\n=== {name.upper()} | Missingness per parking (hoog -> laag) ===")
    per_parking = (
        df.assign(_row_missing=df.isna().mean(axis=1))
        .groupby("parking_id")["_row_missing"]
        .mean()
        .mul(100)
        .round(2)
        .sort_values(ascending=False)
        .rename("avg_row_missing_pct")
    )
    display(per_parking.to_frame())

    print(f"\n=== {name.upper()} | Missingness per featuregroep ===")
    display(feature_group_missingness(df).set_index("feature_group"))


=== SHORTTERM | Overall missingness (top 20) ===


,missing_pct
national_holiday_name,97.16
other_holiday_name,96.68
football_kickoff_hour,96.18
vacation_type,75.83
vacation_name,75.83
humidity_suspect,14.22
qc_precip,14.22
humidity_pct,14.22
sun_duration_min,14.22
shortwave_wm2,14.22



=== SHORTTERM | Missingness per jaar (gemiddeld % missende velden per rij) ===


,avg_row_missing_pct
year,
2018,42.42
2019,34.77
2020,6.49
2021,6.68
2022,6.26
2023,6.83
2024,6.60
2025,6.48
2026,33.57



=== SHORTTERM | Missingness per parking (hoog -> laag) ===


...

,avg_row_missing_pct
year,
2024,6.59



=== LONGTERM | Missingness per parking (hoog -> laag) ===


,avg_row_missing_pct
parking_id,
P Tinel,6.97
P Hoogstraat,6.64
P Kathedraal,6.60
P Keerdok,6.60
P Grote Markt,6.59
P Maarten,6.53
P Komet,6.52



=== LONGTERM | Missingness per featuregroep ===


,missing_pct,n_present_columns,missing_columns
feature_group,,,
calendar,36.899666,9,
events,6.865369,14,
temporal,0.000000,8,
occupancy,0.000000,3,
weather,0.000000,18,
spatial,0.000000,8,
quality_flags,0.000000,5,flag_overcapacity


**Interpretatie**

De hoogste missingness zit vooral in naamvelden die semantisch vaak niet van toepassing zijn (`national_holiday_name`, `other_holiday_name`, `football_kickoff_hour`, vakantie-naam/type). Dit is geen klassieke meetfout, maar wel relevant voor feature-encodering.

In shortterm is missingness duidelijk hoger in de randen van de tijdreeks (2018 en 2026), wat consistent is met opstart- en afkapperiodes.

**Hypothese status**
- H3: Missingness is uniform in de tijd en over parkings. **Verworpen**.

**Volgende logische stap**
- Datakwaliteit en flags kwantificeren, daarna direct extra diagnostiek op verdachte coveragepatronen.

## 4. Datakwaliteit en flag-audit

In [5]:
def as_flag(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series, errors="coerce").fillna(0).gt(0)


def quality_metrics(df: pd.DataFrame) -> pd.Series:
    occupied = pd.to_numeric(df["number_of_occupied_spaces"], errors="coerce")
    capacity = pd.to_numeric(df["number_of_spaces_override"], errors="coerce")
    occ_rate = pd.to_numeric(df["occupancy_rate"], errors="coerce")

    outside_range = occ_rate.lt(0) | occ_rate.gt(1)
    derived_overcapacity = occupied.gt(capacity) & occupied.notna() & capacity.notna()
    derived_inconsistent = occupied.lt(0) | capacity.lt(0) | derived_overcapacity

    metrics = {
        "n_rows": len(df),
        "occupancy_rate_outside_0_1": int(outside_range.sum()),
        "derived_overcapacity": int(derived_overcapacity.sum()),
        "derived_inconsistent": int(derived_inconsistent.sum()),
        "flag_occ_inconsistent": int(as_flag(df["flag_occ_inconsistent"]).sum()),
        "flag_frozen_sensor": int(as_flag(df["flag_frozen_sensor"]).sum()),
        "low_data_coverage": int(as_flag(df["low_data_coverage"]).sum()),
        "system_blackout": int(as_flag(df["system_blackout"]).sum()),
        "partial_year": int(as_flag(df["partial_year"]).sum()),
    }

    if "flag_overcapacity" in df.columns:
        metrics["flag_overcapacity"] = int(as_flag(df["flag_overcapacity"]).sum())
    else:
        metrics["flag_overcapacity"] = np.nan

    result = pd.Series(metrics)
    for key in [
        "occupancy_rate_outside_0_1",
        "derived_overcapacity",
        "derived_inconsistent",
        "flag_occ_inconsistent",
        "flag_frozen_sensor",
        "low_data_coverage",
        "system_blackout",
        "partial_year",
        "flag_overcapacity",
    ]:
        if pd.notna(result.get(key)):
            result[f"{key}_pct"] = float(result[key] / len(df) * 100)
    return result


quality_summary = pd.DataFrame({name: quality_metrics(df) for name, df in dfs.items()}).T
display(quality_summary)

for name, df in dfs.items():
    qc_cols = [c for c in df.columns if c.startswith("qc_")]
    qc_table = []
    for col in qc_cols:
        vc = df[col].value_counts(dropna=False)
        qc_table.append(
            {
                "column": col,
                "dtype": str(df[col].dtype),
                "missing_pct": float(df[col].isna().mean() * 100),
                "true_count": int(vc.get(True, 0)),
                "false_count": int(vc.get(False, 0)),
            }
        )

    print(f"\n=== {name.upper()} | QC-kolommen ===")
    display(pd.DataFrame(qc_table).set_index("column"))

,derived_inconsistent,derived_inconsistent_pct,derived_overcapacity,derived_overcapacity_pct,flag_frozen_sensor,flag_frozen_sensor_pct,flag_occ_inconsistent,flag_occ_inconsistent_pct,flag_overcapacity,flag_overcapacity_pct,low_data_coverage,low_data_coverage_pct,n_rows,occupancy_rate_outside_0_1,occupancy_rate_outside_0_1_pct,partial_year,partial_year_pct,system_blackout,system_blackout_pct
shortterm,0.0,0.0,0.0,0.0,24392.0,8.572915,0.0,0.0,0.0,0.0,26397.0,9.2776,284524.0,0.0,0.0,7690.0,2.70276,2564.0,0.901154
longterm,0.0,0.0,0.0,0.0,4112.0,8.815899,0.0,0.0,NaN,NaN,0.0,0.0000,46643.0,0.0,0.0,0.0,0.00000,0.0,0.000000



=== SHORTTERM | QC-kolommen ===


,dtype,missing_pct,true_count,false_count
column,,,,
qc_temp,object,14.218484,242714,1355
qc_precip,object,14.218484,243055,1014
qc_wind_speed,object,14.218484,243215,854
qc_wind_gusts,object,14.218484,243209,860
qc_humidity,object,14.218484,188662,55407
qc_pressure,object,14.218484,243251,818



=== LONGTERM | QC-kolommen ===


,dtype,missing_pct,true_count,false_count
column,,,,
qc_temp,bool,0.0,46538,105
qc_precip,bool,0.0,46590,53
qc_wind_speed,bool,0.0,46538,105
qc_wind_gusts,bool,0.0,46538,105
qc_humidity,bool,0.0,44166,2477
qc_pressure,bool,0.0,46489,154


**Interpretatie**

Er zijn geen observaties met `occupancy_rate` buiten [0,1] en er zijn geen overcapacity-gevallen in de huidige extracten. Dat is positief voor basale plausibiliteit.

Tegelijk is `flag_frozen_sensor` substantieel aanwezig in beide datasets (ongeveer 8-9%), en shortterm bevat bijkomend `low_data_coverage` en `system_blackout`-gevallen. Dit vraagt om expliciete sensitiviteitsanalyses met striktere filters.

**Hypothese status**
- H4: Ernstige fysische onmogelijkheid in occupancy-metrieken komt frequent voor. **Verworpen**.
- H5: Operationele datakwaliteitsrisico's (frozen/coverage) zijn niet verwaarloosbaar. **Ondersteund**.

**Volgende logische stap**
- Extra diagnostiek op de verdachte patronen (tijdsgaten en parking-specifieke coverage), vervolgens formeel coverage-overzicht.

## 5. Extra diagnostiek op verdachte patronen

In [6]:
def time_gap_profile(df: pd.DataFrame) -> pd.DataFrame:
    out = []
    for parking_id, grp in df.sort_values(["parking_id", "rounded_hour"]).groupby("parking_id"):
        ts = pd.to_datetime(grp["rounded_hour"], errors="coerce").dropna().sort_values()
        if len(ts) < 2:
            out.append({"parking_id": parking_id, "max_gap_h": np.nan, "p95_gap_h": np.nan, "pct_gaps_gt_1h": np.nan})
            continue

        gaps_h = ts.diff().dropna().dt.total_seconds().div(3600)
        out.append(
            {
                "parking_id": parking_id,
                "max_gap_h": float(gaps_h.max()),
                "p95_gap_h": float(gaps_h.quantile(0.95)),
                "pct_gaps_gt_1h": float((gaps_h > 1).mean() * 100),
            }
        )
    return pd.DataFrame(out).sort_values(["max_gap_h", "pct_gaps_gt_1h"], ascending=False)


gap_profiles = {name: time_gap_profile(df) for name, df in dfs.items()}
for name, gap_df in gap_profiles.items():
    print(f"\n=== {name.upper()} | Grootste tijdsgaten per parking ===")
    display(gap_df)

lt = dfs["longterm"].copy()
if "P Tinel" in set(lt["parking_id"].dropna().unique()):
    lt_tinel = lt.loc[lt["parking_id"] == "P Tinel"].copy()
    lt_tinel["month_period"] = lt_tinel["rounded_hour"].dt.to_period("M").astype(str)

    tinel_monthly_obs = lt_tinel.groupby("month_period").size().rename("n_obs")
    full_2024 = pd.date_range("2024-01-01 00:00:00", "2024-12-31 23:00:00", freq="h")
    covered_hours = lt_tinel["rounded_hour"].nunique()
    coverage_pct_2024 = covered_hours / len(full_2024) * 100

    print("\n=== LONGTERM | Diagnostiek P Tinel ===")
    print(f"n_obs: {len(lt_tinel):,}")
    print(f"covered unique hours in 2024: {covered_hours:,} / {len(full_2024):,} ({coverage_pct_2024:.2f}%)")
    print(f"date_min: {lt_tinel['rounded_hour'].min()} | date_max: {lt_tinel['rounded_hour'].max()}")
    display(tinel_monthly_obs.to_frame())


def longest_constant_run(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for parking_id, grp in df.sort_values(["parking_id", "rounded_hour"]).groupby("parking_id"):
        g = grp[["rounded_hour", "number_of_occupied_spaces", "flag_frozen_sensor"]].copy()
        g["delta_h"] = g["rounded_hour"].diff().dt.total_seconds().div(3600)
        same_occ = g["number_of_occupied_spaces"].eq(g["number_of_occupied_spaces"].shift())
        contiguous = g["delta_h"].eq(1)
        new_run = ~(same_occ & contiguous)
        run_id = new_run.cumsum()
        run_lengths = g.groupby(run_id).size()

        rows.append(
            {
                "parking_id": parking_id,
                "longest_constant_run_h": int(run_lengths.max()),
                "n_runs_ge_12h": int((run_lengths >= 12).sum()),
                "n_runs_ge_24h": int((run_lengths >= 24).sum()),
                "frozen_flag_pct": float(as_flag(g["flag_frozen_sensor"]).mean() * 100),
            }
        )
    return pd.DataFrame(rows).sort_values(["frozen_flag_pct", "longest_constant_run_h"], ascending=False)


for name, df in dfs.items():
    print(f"\n=== {name.upper()} | Frozen-diagnostiek via constante runs ===")
    display(longest_constant_run(df))


=== SHORTTERM | Grootste tijdsgaten per parking ===


,parking_id,max_gap_h,p95_gap_h,pct_gaps_gt_1h
1,P Grote Markt,13545.0,3.0,7.438332
3,P Kathedraal,10033.0,2.0,6.222383
8,P Tinel,9924.0,3.0,7.955181
6,P Lamot,9922.0,1.0,4.822356
0,P Bruul,6791.0,1.0,1.798731
9,P Veemarkt,6722.0,3.0,8.756255
2,P Hoogstraat,1841.0,1.0,3.082466
7,P Maarten,12.0,1.0,0.051706
5,P Komet,11.0,1.0,0.023708
4,P Keerdok,2.0,1.0,1.328878



=== LONGTERM | Grootste tijdsgaten per parking ===


,parking_id,max_gap_h,p95_gap_h,pct_gaps_gt_1h
6,P Tinel,942.0,31.0,11.295681
1,P Hoogstraat,180.0,1.0,0.565904
0,P Grote Markt,38.0,1.0,2.325856
5,P Maarten,10.0,1.0,0.218598
4,P Komet,9.0,1.0,0.419393
2,P Kathedraal,7.0,1.0,0.183360
3,P Keerdok,6.0,1.0,0.080009



=== LONGTERM | Diagnostiek P Tinel ===
n_obs: 302
covered unique hours in 2024: 302 / 8,784 (3.44%)
date_min: 2024-07-26 00:00:00 | date_max: 2024-12-11 05:00:00


,n_obs
month_period,
2024-07,8
2024-08,47
2024-09,168
2024-10,38
2024-12,41



=== SHORTTERM | Frozen-diagnostiek via constante runs ===


,parking_id,longest_constant_run_h,n_runs_ge_12h,n_runs_ge_24h,frozen_flag_pct
7,P Maarten,484,109,44,37.355393
5,P Komet,169,87,13,26.376926
0,P Bruul,1850,153,10,17.660961
6,P Lamot,1850,8,4,6.616862
3,P Kathedraal,31,26,1,5.203120
2,P Hoogstraat,46,14,1,4.670195
8,P Tinel,19,12,0,4.386012
4,P Keerdok,23,4,0,3.295493
9,P Veemarkt,19,3,0,1.860334
1,P Grote Markt,102,7,1,1.203307



=== LONGTERM | Frozen-diagnostiek via constante runs ===


,parking_id,longest_constant_run_h,n_runs_ge_12h,n_runs_ge_24h,frozen_flag_pct
5,P Maarten,462,30,7,29.388030
6,P Tinel,37,1,1,13.245033
1,P Hoogstraat,66,19,10,10.998468
4,P Komet,13,1,0,8.168400
2,P Kathedraal,37,3,1,5.305374
0,P Grote Markt,11,0,0,3.151930
3,P Keerdok,10,0,0,1.988571


**Interpretatie**

De gap-analyse toont sterke heterogeniteit per parking, met zeer grote maximale gaten in shortterm en een uitgesproken coverageprobleem voor `P Tinel` in longterm (zeer weinig observaties in 2024).

De run-diagnostiek bevestigt dat lange constante bezettingsreeksen bestaan, wat inhoudelijk past bij `flag_frozen_sensor` als operationeel risico.

**Hypothese status**
- H6: Coverage is homogeen over parkings. **Verworpen**.
- H7: Frozen-sensor signalen zijn consistent met sequentiegedrag. **Voorlopig ondersteund**.

**Volgende logische stap**
- Formeel coverage-overzicht op datasetniveau en vervolgens een expliciet EDA-protocol voor full vs strict subsets.

## 6. Coverage-overzicht

In [7]:
coverage_outputs = {}

for name, df in dfs.items():
    obs_per_year = df.groupby("year").size().rename("n_obs").to_frame()
    obs_per_parking = df.groupby("parking_id").size().sort_values(ascending=False).rename("n_obs").to_frame()
    hour_coverage = df.groupby("parking_id")["hour"].nunique().rename("n_unique_hours").to_frame()
    hour_distribution = df.groupby("hour").size().rename("n_obs").to_frame()

    coverage_outputs[name] = {
        "obs_per_year": obs_per_year,
        "obs_per_parking": obs_per_parking,
        "hour_coverage": hour_coverage,
        "hour_distribution": hour_distribution,
        "gap_profile": gap_profiles[name].set_index("parking_id"),
    }

for name, obj in coverage_outputs.items():
    print(f"\n=== {name.upper()} | Observaties per jaar ===")
    display(obj["obs_per_year"])

    print(f"\n=== {name.upper()} | Observaties per parking ===")
    display(obj["obs_per_parking"])

    print(f"\n=== {name.upper()} | Uurdekking per parking ===")
    display(obj["hour_coverage"])

    print(f"\n=== {name.upper()} | Uurverdeling (globaal) ===")
    display(obj["hour_distribution"])

    print(f"\n=== {name.upper()} | Gaten in tijdreeksen ===")
    display(obj["gap_profile"])


=== SHORTTERM | Observaties per jaar ===


,n_obs
year,
2018,5
2019,33000
2020,32741
2021,13630
2022,12762
2023,39980
2024,57116
2025,87600
2026,7690



=== SHORTTERM | Observaties per parking ===


,n_obs
parking_id,
P Bruul,40196
P Tinel,37574
P Grote Markt,36649
P Lamot,35606
P Veemarkt,35370
P Keerdok,26339
P Hoogstraat,22483
P Kathedraal,22179
P Maarten,15473



=== SHORTTERM | Uurdekking per parking ===


...

,n_unique_hours
parking_id,
P Grote Markt,24
P Hoogstraat,24
P Kathedraal,24
P Keerdok,24
P Komet,24
P Maarten,24
P Tinel,24



=== LONGTERM | Uurverdeling (globaal) ===


,n_obs
hour,
0,1925
1,1909
2,1918
3,1922
4,1922
5,1944
6,1958
7,1956
8,1907



=== LONGTERM | Gaten in tijdreeksen ===


,max_gap_h,p95_gap_h,pct_gaps_gt_1h
parking_id,,,
P Tinel,942.0,31.0,11.295681
P Hoogstraat,180.0,1.0,0.565904
P Grote Markt,38.0,1.0,2.325856
P Maarten,10.0,1.0,0.218598
P Komet,9.0,1.0,0.419393
P Kathedraal,7.0,1.0,0.183360
P Keerdok,6.0,1.0,0.080009


**Interpretatie**

Coverage op uurniveau lijkt op het eerste gezicht volledig (24/24 uur aanwezig per parking), maar de gap-profielen tonen dat dit geen continue tijdreeks garandeert. Voor inferentie over dynamiek moeten we daarom niet enkel op `hour`-dekking vertrouwen, maar ook op temporele continuiteit en kwaliteitsflags.

**Hypothese status**
- H8: Volledige uurset impliceert voldoende temporele dekking. **Verworpen**.

**Volgende logische stap**
- Definieer expliciet een EDA-protocol met standaard in-scope regels, flagrapportering en sensitivity-subsets.

## 7. Expliciet EDA-analyseprotocol

In [8]:
def build_subset_masks(df: pd.DataFrame) -> dict[str, pd.Series]:
    base = pd.Series(True, index=df.index)

    in_range = pd.to_numeric(df["occupancy_rate"], errors="coerce").between(0, 1, inclusive="both")

    hard_quality_ok = base.copy()
    for col in ["system_blackout", "low_data_coverage", "partial_year", "flag_occ_inconsistent"]:
        if col in df.columns:
            hard_quality_ok &= ~as_flag(df[col])

    strict_ok = hard_quality_ok.copy()
    if "flag_frozen_sensor" in df.columns:
        strict_ok &= ~as_flag(df["flag_frozen_sensor"])

    masks = {
        "full_descriptive": base,
        "quality_filtered": hard_quality_ok & in_range,
        "strict_inference": strict_ok & in_range,
    }
    return masks


subset_summary_rows = []
for name, df in dfs.items():
    masks = build_subset_masks(df)
    for subset_name, mask in masks.items():
        subset_summary_rows.append(
            {
                "dataset": name,
                "subset": subset_name,
                "n_obs": int(mask.sum()),
                "pct_of_dataset": float(mask.mean() * 100),
            }
        )

subset_summary = pd.DataFrame(subset_summary_rows).sort_values(["dataset", "subset"])
display(subset_summary)

,dataset,subset,n_obs,pct_of_dataset
3,longterm,full_descriptive,46643,100.000000
4,longterm,quality_filtered,46643,100.000000
5,longterm,strict_inference,42531,91.184101
0,shortterm,full_descriptive,284524,100.000000
1,shortterm,quality_filtered,250437,88.019640
2,shortterm,strict_inference,227688,80.024181


### Protocolkeuzes voor verdere EDA

**Standaard in scope**
- `full_descriptive`: volledige dataset voor beschrijvende frequenties, basisplots en representativiteitschecks.
- `quality_filtered`: sluit `system_blackout`, `low_data_coverage`, `partial_year` en `flag_occ_inconsistent` uit.
- `strict_inference`: idem als `quality_filtered` + uitsluiten van `flag_frozen_sensor`.

**Flags die standaard gerapporteerd worden (per analyse, per dataset, per parking indien relevant)**
- `flag_frozen_sensor`
- `low_data_coverage`
- `system_blackout`
- `partial_year`
- `flag_occ_inconsistent`
- `flag_overcapacity` (enkel shortterm; in longterm afgeleid via occupancy > capacity)

**Welke analyses op welke subset?**
- Beschrijvend (niveau, spreiding, algemene seizoenspatronen): eerst `full_descriptive`, daarna robuustheidscheck met `quality_filtered`.
- Inference-gevoelige analyses (effectschattingen, correlaties voor featureselectie, hypothesetoetsing): primair `strict_inference`, met vergelijking tegen `quality_filtered`.
- Parking-specifieke of event-analyses: altijd coverage-context meegeven (observaties, gaps, flag burden) om selectiebias expliciet te maken.

**Gevoeligheidsanalyses (verplicht in volgende notebooks)**
- Herhaal kernresultaten met en zonder `flag_frozen_sensor`.
- Voor shortterm: herhaal analyses met en zonder `partial_year` en `low_data_coverage`.
- Voor longterm: aparte robuustheidsrapportering voor parkings met beperkte dekking (met name `P Tinel`).

## Data audit conclusions
De data zijn bruikbaar als EDA-basis, met sterk vergelijkbare schema's tussen shortterm en longterm. Er zijn geen evidente fysisch onmogelijke occupancy-waarden, maar wel duidelijke operationele kwaliteitsrisico's (vooral `flag_frozen_sensor`, en in shortterm ook `low_data_coverage`/`system_blackout`/`partial_year`). Coverage is bovendien heterogeen per parking, met uitgesproken fragmentatie voor bepaalde reeksen.

## Risks for interpretation
1. Tijdsfragmentatie kan trend- en seizoensinterpretaties vertekenen, zelfs wanneer alle uren (0-23) voorkomen.
2. Hoge missingness in naamvelden (holiday/event labels) is grotendeels structureel, maar kan foutief behandeld worden als random missingness.
3. Parking-specifieke coverageverschillen (bv. `P Tinel` longterm) verhogen het risico op selectiebias in cross-parking vergelijkingen.
4. Frozen-sensor flags impliceren mogelijk meetplateaus; analyses die op korte-termijnvariatie steunen zijn hiervoor gevoelig.

## Implications for downstream EDA and feature engineering
1. Bouw elke volgende notebook met dual reporting: `full_descriptive` + `strict_inference`.
2. Voeg kwaliteitsflags expliciet toe als features en/of filtercriteria, en rapporteer hun prevalentie systematisch.
3. Gebruik coverage-metadata (gaps, observatievolume per parking) als randvoorwaarde bij interpretatie en modelvalidatie.
4. Vermijd leakage door kwaliteitssignalen enkel te gebruiken wanneer ze ook operationeel beschikbaar zijn op voorspeltijdstip.